# Olist Logistic Agent

## Exploratório de Dados: Capacidade da Malha Logística e Diagnóstico de Atrasos
> **Projeto:** Transformação Digital Olist via IA Agêntica  
> **Foco do Notebook:** Cruzamento de relatórios de pedidos, entregas, geolocalização e performance de vendedores.

---

### Objetivo deste Notebook
Este estudo de dados aplica um método de eliminação de hipóteses para identificar os reais mecanismos dominantes por trás dos atrasos, fundamentando a arquitetura do **Agente de Performance Logística**.

Neste notebook, realizamos:
1. **Validação de Promessas de Entrega:** Análise de probabilidade de cumprimento de prazo por região geográfica (ex: Nordeste vs. Sudeste).
2. **Desempenho por Origem e Sazonalidade:** Investigação de gargalos em estados de origem e comparação entre picos comerciais (ex: Black Friday vs. períodos sem apelo promocional).
3. **Análise de Correlação e SLA do Vendedor:** Avaliação do impacto da taxa de postagem fora do prazo pelo vendedor no prazo final de entrega e identificação da receita exposta a riscos de cancelamento.

## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Carregar dados relevantes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PASTA = Path('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/')

ordens_full = pd.read_csv(PASTA / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_carrier_date',
    'order_delivered_customer_date', 'order_estimated_delivery_date'])

itens_sl = pd.read_csv(PASTA / 'olist_order_items_dataset.csv',
                       parse_dates=['shipping_limit_date'])
limite = itens_sl.groupby('order_id')['shipping_limit_date'].max().rename('limite')

log = (ordens_full[ordens_full['order_delivered_customer_date'].notna()]
       .merge(limite, on='order_id', how='left'))

log['mes'] = log['order_purchase_timestamp'].dt.to_period('M').dt.to_timestamp()
log['dias_atraso'] = (log['order_delivered_customer_date']
                      - log['order_estimated_delivery_date']).dt.days
log['atrasado'] = log['dias_atraso'] > 0
log['postou_atrasado'] = log['order_delivered_carrier_date'] > log['limite']
log['dias_transito'] = (log['order_delivered_customer_date']
                        - log['order_delivered_carrier_date']).dt.days
log['dias_postagem'] = (log['order_delivered_carrier_date']
                        - log['order_purchase_timestamp']).dt.days
log['dias_total'] = (log['order_delivered_customer_date']
                     - log['order_purchase_timestamp']).dt.days

## Analise de reclamações onde identificamos que muita elas estão ligadas a logística.

A média de atraso entre as reclamações é 31%. Acima da linha, é entrega; abaixo, é produto.

69% das reclamações não tiveram atraso nenhum. Zerar o atraso não resolveria a maior parte delas.

Esse número faz duas coisas ao mesmo tempo:

Ele dimensiona o invisível. Defeito, produto errado, qualidade abaixo e entrega parcial não aparecem em nenhum indicador operacional — o pedido consta entregue, no prazo.
Ele prova que um Agente de Reviews não é redundante com um de Logística. São populações majoritariamente disjuntas. Sem esse teste, propor dois agentes seria indefensável.

In [ ]:
# ══════════ G9 — % de atraso por tipo de reclamação ══════════
rec = reclamacoes.merge(
    ent[['order_id', 'atrasado', 'order_delivered_customer_date']],
    on='order_id', how='left')
rec_ent = rec[rec['order_delivered_customer_date'].notna()]

perfil = rec_ent.groupby('tipo').agg(
    n=('review_id', 'size'), pct_atraso=('atrasado', 'mean'))
perfil['pct_atraso'] *= 100
perfil = perfil.sort_values('pct_atraso')
media = rec_ent['atrasado'].mean()*100

fig, ax = plt.subplots(figsize=(10.5, 5.4))
cores = [VERMELHO if v > media else AZUL for v in perfil['pct_atraso']]
ax.barh(range(len(perfil)), perfil['pct_atraso'], color=cores, height=0.68)

for i, (v, n_) in enumerate(zip(perfil['pct_atraso'], perfil['n'])):
    ax.annotate(f'{v:.0f}%   ({n_:,} casos)'.replace(',', '.'), xy=(v, i),
                xytext=(6, 0), textcoords='offset points', va='center',
                fontsize=8.8, color='#4a535f')

ax.axvline(media, color=CINZA, linewidth=1.4, linestyle='--', zorder=4)
ax.annotate(f'média: {media:.0f}%', xy=(media, len(perfil)-0.3), xytext=(6, 0),
            textcoords='offset points', color=CINZA, fontsize=8.5)

ax.set_yticks(range(len(perfil)))
ax.set_yticklabels(perfil.index, fontsize=9.5)
ax.set_title('Quais reclamações são de logística — e quais não são')
ax.set_xlabel('% das reclamações daquele tipo que tiveram atraso')
ax.set_xlim(0, perfil['pct_atraso'].max()*1.55)
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

A promessa e a variância
Mediana ponta a ponta: 10,3 dias — aprovação em horas, postagem 1,8 d, trânsito 7,1 d (69% do tempo total).

A hipótese intuitiva era que os 23 dias prometidos são gordura. Refutada: o P90 real é 23,1 dias. A promessa está estatisticamente correta — ela cobre 90% dos casos, que é exatamente o que uma promessa deve fazer.

O erro não é o tamanho da promessa. É prometer igual onde o risco não é igual.

In [ ]:
# ══════════ G10 — Prazo prometido vs P90 real, por região ══════════
REGIAO = {**dict.fromkeys(['AC','AP','AM','PA','RO','RR','TO'], 'Norte'),
          **dict.fromkeys(['AL','BA','CE','MA','PB','PE','PI','RN','SE'], 'Nordeste'),
          **dict.fromkeys(['DF','GO','MT','MS'], 'Centro-Oeste'),
          **dict.fromkeys(['ES','MG','RJ','SP'], 'Sudeste'),
          **dict.fromkeys(['PR','RS','SC'], 'Sul')}

e = ent.merge(customers[['customer_id', 'customer_state']], on='customer_id')
e['regiao'] = e['customer_state'].map(REGIAO)
e['real'] = (e['order_delivered_customer_date']
             - e['order_purchase_timestamp']).dt.days
e['prometido'] = (e['order_estimated_delivery_date']
                  - e['order_purchase_timestamp']).dt.days

g = e.groupby('regiao').agg(
    prometido=('prometido', 'median'),
    p90_real=('real', lambda s: s.quantile(0.9)),
    pct_atraso=('atrasado', 'mean'))

# CORREÇÃO: converter para % ANTES de arredondar
g['pct_atraso'] = g['pct_atraso'] * 100
g = g.round(1).sort_values('p90_real', ascending=False)

x = np.arange(len(g)); larg = 0.36
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - larg/2, g['prometido'], larg, color=AZUL, label='Prazo prometido')
b2 = ax.bar(x + larg/2, g['p90_real'], larg, color=LARANJA, label='P90 da entrega real')

for bb in list(b1) + list(b2):
    ax.annotate(f'{bb.get_height():.0f}', xy=(bb.get_x()+bb.get_width()/2, bb.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center',
                fontsize=9, fontweight='700')

for i, (reg, linha) in enumerate(g.iterrows()):
    if linha['p90_real'] > linha['prometido']:
        ax.annotate('promessa\ninsustentável', xy=(i, linha['p90_real']),
                    xytext=(0, 26), textcoords='offset points', ha='center',
                    fontsize=8.5, color=VERMELHO, fontweight='700', linespacing=1.3)

ax.set_xticks(x); ax.set_xticklabels(g.index, fontsize=10)
ax.set_title('Onde o P90 real supera o prometido, a promessa não se sustenta')
ax.set_ylabel('Dias'); ax.set_ylim(0, g['p90_real'].max()*1.32)
ax.legend(frameon=False, fontsize=9.5); ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

print(g.to_string())
print('\n→ Use os valores acima na seção 6. A coluna pct_atraso agora mostra a taxa'
      '\n  real de atraso por região, e não 10,0% em todas.')

In [ ]:
# Receita mensal (mesma base do G1 do relatório)
itens = order_items.merge(
    orders[['order_id', 'order_status', 'order_purchase_timestamp']], on='order_id')
itens = itens[~itens['order_status'].isin(['canceled', 'unavailable'])]
itens['mes'] = itens['order_purchase_timestamp'].dt.to_period('M').dt.to_timestamp()
receita = itens.groupby('mes')['price'].sum()['2017-01':'2018-08']

# Série mensal de atraso e de postagem fora do prazo
mm = log.groupby('mes').agg(
    pct_atraso=('atrasado', 'mean'),
    pct_postagem=('postou_atrasado', 'mean'))['2017-01':'2018-08'] * 100

print('═══ NÚMEROS PARA FIXAR NO RELATÓRIO ═══')
print(f"Atraso médio da plataforma: {log['atrasado'].mean()*100:.1f}%")
print(f"Mediana ponta a ponta: {log['dias_total'].median():.1f} dias")
print(f"P90 da entrega real: {log['dias_total'].quantile(0.90):.1f} dias")
print(f"Mediana postagem: {log['dias_postagem'].median():.1f} d · "
      f"trânsito: {log['dias_transito'].median():.1f} d")
print(f"Correlação receita x atraso: r = {receita.corr(mm['pct_atraso']):+.2f}")
print('\nSérie mensal (use estes valores no texto):')
print(mm.round(1).to_string())


# ══════════ GRÁFICO 1 — Os picos de venda coincidem com os picos de atraso ══════════
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True,
                             gridspec_kw={'hspace': 0.15})

a1.plot(receita.index, receita.values, color=AZUL, linewidth=2)
a1.fill_between(receita.index, receita.values, color=AZUL, alpha=0.10)
a1.set_ylabel('Receita')
a1.set_ylim(0, receita.max() * 1.2)
a1.yaxis.set_major_formatter(plt.FuncFormatter(brl))
a1.set_title('Os picos de venda coincidem com os picos de atraso')
a1.grid(axis='x', visible=False)

a2.plot(mm.index, mm['pct_atraso'], color=LARANJA, linewidth=2,
        label='% de pedidos atrasados')
a2.set_ylabel('% dos pedidos')
a2.set_ylim(0, mm['pct_atraso'].max() * 1.35)
a2.legend(frameon=False, loc='upper left', fontsize=9)
a2.grid(axis='x', visible=False)

r = receita.corr(mm['pct_atraso'])
a2.annotate(f'correlação receita × atraso: r = {r:+.2f}', xy=(0.99, 0.08),
            xycoords='axes fraction', ha='right', fontweight='700', fontsize=9.5)

plt.tight_layout()
plt.show()


# ══════════ GRÁFICO 2 — Prazo prometido vs P90 real, por região ══════════
REGIAO = {**dict.fromkeys(['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'], 'Norte'),
          **dict.fromkeys(['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'], 'Nordeste'),
          **dict.fromkeys(['DF', 'GO', 'MT', 'MS'], 'Centro-Oeste'),
          **dict.fromkeys(['ES', 'MG', 'RJ', 'SP'], 'Sudeste'),
          **dict.fromkeys(['PR', 'RS', 'SC'], 'Sul')}

e = log.merge(customers[['customer_id', 'customer_state']], on='customer_id')
e['regiao'] = e['customer_state'].map(REGIAO)
e['prometido'] = (e['order_estimated_delivery_date']
                  - e['order_purchase_timestamp']).dt.days

gr = e.groupby('regiao').agg(
    prometido=('prometido', 'median'),
    p90_real=('dias_total', lambda s: s.quantile(0.9)),
    pct_atraso=('atrasado', 'mean')).round(1)
gr['pct_atraso'] = (gr['pct_atraso'] * 100).round(1)
gr = gr.sort_values('p90_real', ascending=False)

x = np.arange(len(gr))
larg = 0.36
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - larg/2, gr['prometido'], larg, color=AZUL, label='Prazo prometido')
b2 = ax.bar(x + larg/2, gr['p90_real'], larg, color=LARANJA, label='P90 da entrega real')

for bb in list(b1) + list(b2):
    ax.annotate(f'{bb.get_height():.0f}',
                xy=(bb.get_x() + bb.get_width()/2, bb.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center',
                fontsize=9, fontweight='700')

for i, (reg, linha) in enumerate(gr.iterrows()):
    if linha['p90_real'] > linha['prometido']:
        ax.annotate('promessa\ninsustentável', xy=(i, linha['p90_real']),
                    xytext=(0, 26), textcoords='offset points', ha='center',
                    fontsize=8.5, color=VERMELHO, fontweight='700', linespacing=1.3)

ax.set_xticks(x)
ax.set_xticklabels(gr.index, fontsize=10)
ax.set_title('Onde o P90 real supera o prometido, a promessa não se sustenta')
ax.set_ylabel('Dias')
ax.set_ylim(0, gr['p90_real'].max() * 1.32)
ax.legend(frameon=False, fontsize=9.5)
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

print(gr.to_string())


# ══════════ GRÁFICO 3 — Não existe estado que atrasa ══════════
sellers = pd.read_csv(PASTA / 'olist_sellers_dataset.csv')

origem = (order_items[['order_id', 'seller_id']]
          .drop_duplicates('order_id')
          .merge(sellers[['seller_id', 'seller_state']], on='seller_id'))

uf = log.merge(origem, on='order_id').groupby('seller_state').agg(
    n=('atrasado', 'size'), pct=('atrasado', 'mean'))
uf['pct'] *= 100
uf = uf[uf['n'] >= 500].sort_values('pct')          # só UFs com volume relevante
media_uf = log['atrasado'].mean() * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(range(len(uf)), uf['pct'], color=AZUL, height=0.65)

for i, (v, n_) in enumerate(zip(uf['pct'], uf['n'])):
    ax.annotate(f'{v:.1f}%   ({n_:,} pedidos)'.replace(',', '.'), xy=(v, i),
                xytext=(6, 0), textcoords='offset points', va='center',
                fontsize=8.8, color='#4a535f')

ax.axvline(media_uf, color=CINZA, linewidth=1.4, linestyle='--', zorder=4)
ax.annotate(f'média da plataforma: {media_uf:.1f}%',
            xy=(media_uf, len(uf) - 0.4), xytext=(6, 0),
            textcoords='offset points', color=CINZA, fontsize=8.5)

ax.set_yticks(range(len(uf)))
ax.set_yticklabels(uf.index, fontsize=9.5)
ax.set_title('Não existe "estado que atrasa" — a dispersão entre origens é pequena')
ax.set_xlabel('% de pedidos atrasados por estado de origem do vendedor')
ax.set_xlim(0, uf['pct'].max() * 1.45)
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()


# ══════════ GRÁFICO 4 — Duas crises, causas diferentes ══════════
fig, ax = plt.subplots(figsize=(10.5, 5))
ax.plot(mm.index, mm['pct_atraso'], color=VERMELHO, linewidth=2,
        label='% de pedidos atrasados', zorder=3)
ax.plot(mm.index, mm['pct_postagem'], color=LARANJA, linewidth=2,
        label='% de postagens fora do prazo (vendedor)', zorder=3)

for ts, txt in [('2017-11-01', 'Black Friday\nvendedor sobrecarregado'),
                ('2018-03-01', 'mar/2018\ngargalo no transporte')]:
    t = pd.Timestamp(ts)
    v = mm.loc[t, 'pct_atraso']
    ax.scatter([t], [v], s=60, color=VERMELHO, edgecolor='white',
               linewidth=2, zorder=5)
    ax.annotate(txt, xy=(t, v), xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=8.5, fontweight='700', linespacing=1.3)

ax.set_title('Duas crises, causas diferentes — só uma foi sazonal')
ax.set_ylabel('% dos pedidos')
ax.set_ylim(0, mm.max().max() * 1.4)
ax.legend(frameon=False, loc='upper left', fontsize=9)
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

print('\nTabela das duas crises:')
print(mm.loc[['2017-11-01', '2018-03-01']].round(1).to_string())


# ══════════ GRÁFICO 5 — O vendedor é o fator mais forte, mas não é a maior parte ══════════
lp = log[log['postou_atrasado'].notna()].copy()
lp['dias_pos'] = lp['dias_atraso'].clip(lower=0)

taxa = lp.groupby('postou_atrasado')['atrasado'].mean() * 100
dias = lp.groupby('postou_atrasado')['dias_pos'].sum()
share = dias / dias.sum() * 100

rot = ['Postou\nno prazo', 'Postou\nfora do prazo']
fator = taxa[True] / taxa[False]

fig, (p1, p2) = plt.subplots(1, 2, figsize=(11, 4.8))

# Painel esquerdo — a alavanca
p1.bar(rot, [taxa[False], taxa[True]], color=[AZUL, VERMELHO], width=0.55)
for i, v in enumerate([taxa[False], taxa[True]]):
    p1.annotate(f'{v:.1f}%', xy=(i, v), xytext=(0, 5),
                textcoords='offset points', ha='center',
                fontsize=12, fontweight='700')
p1.set_title(f'A alavanca: postar fora do prazo\nmultiplica o atraso por {fator:.1f}×',
             fontsize=11.5)
p1.set_ylabel('% de pedidos com atraso final')
p1.set_ylim(0, taxa.max() * 1.3)
p1.grid(axis='x', visible=False)

# Painel direito — a massa
p2.bar(rot, [share[False], share[True]], color=[CINZA, VERMELHO], width=0.55)
for i, v in enumerate([share[False], share[True]]):
    p2.annotate(f'{v:.0f}%', xy=(i, v), xytext=(0, 5),
                textcoords='offset points', ha='center',
                fontsize=12, fontweight='700')
p2.set_title('A massa: a maior parte dos dias de atraso\nocorre com o vendedor em dia',
             fontsize=11.5)
p2.set_ylabel('% do total de dias de atraso acumulados')
p2.set_ylim(0, 100)
p2.grid(axis='x', visible=False)

plt.tight_layout()
plt.show()

print(f'\nTaxa de atraso — postou no prazo: {taxa[False]:.1f}% · '
      f'fora do prazo: {taxa[True]:.1f}% (fator {fator:.1f}x)')
print(f'Participação nos dias de atraso — no prazo: {share[False]:.0f}% · '
      f'fora do prazo: {share[True]:.0f}%')


# ══════════ GRÁFICO 6 (OPCIONAL) — O custo do atraso na avaliação ══════════
# Use se quiser sustentar a frase "cada atraso custa 2,5 pontos de nota".
rev = order_reviews.drop_duplicates('order_id')[['order_id', 'review_score']]
nota = log.merge(rev, on='order_id', how='inner')
nota['faixa'] = pd.cut(nota['dias_atraso'], [-999, 0, 7, 999],
                       labels=['No prazo', 'Atraso\n1 a 7 dias', 'Atraso\n8+ dias'])

gn = nota.groupby('faixa', observed=True)['review_score'].agg(['mean', 'size'])
pct1 = nota.groupby('faixa', observed=True)['review_score'].apply(
    lambda s: (s == 1).mean() * 100)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(gn.index.astype(str), gn['mean'], color=[AZUL, LARANJA, VERMELHO], width=0.55)
for i, (v, n_, p) in enumerate(zip(gn['mean'], gn['size'], pct1)):
    ax.annotate(f'{v:.2f}\n{p:.0f}% viram nota 1\nn={n_:,}'.replace(',', '.'),
                xy=(i, v), xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=9, fontweight='700', linespacing=1.4)

ax.set_title('O atraso destrói a avaliação — e o efeito não é gradual')
ax.set_ylabel('Nota média da avaliação')
ax.set_ylim(0, 5.8)
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()